In [12]:
import os
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

In [13]:
def download_pdf(pdf_tuple):
    url = pdf_tuple[0]
    filename = pdf_tuple[1] + '.pdf'
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36"
        }
        response = requests.get(url, headers=headers, verify=False)
    except:
        print('Download failed.', pdf_tuple)
        return
    with open(f'pdfs/{pdfs_dir_name}/' + filename, "wb") as f:
        f.write(response.content)


In [14]:
input_file = 'journal_40_oai_dc.xlsx'
input_metadata_path = 'metadata/' + input_file
pdfs_dir_name = input_file.replace('_oai_dc.xlsx', '')

In [15]:
os.mkdir('pdfs/' + pdfs_dir_name)

In [16]:
df = pd.read_excel(input_metadata_path)

pdf_urls = []
for _, row in df.iterrows():
    try:
        pdf_url = row['relation'].split(' | ')
        pdf_url = pdf_url[0]
        pdf_url = pdf_url.replace('/view/', '/download/')

        pdf_name = row['oai_id']
        for ch in [":", "/"]:
            pdf_name = pdf_name.replace(ch, "_")

        pdf_urls.append((pdf_url, pdf_name))
    except:
        with open(f'errors/{pdfs_dir_name}.txt', 'a', encoding='utf-8') as txt:
            txt.write(row['oai_id'] + '\n')


In [17]:
pdf_urls[0]

('https://www.jatjournal.org/index.php/jat/article/download/10/19',
 'oai_ojs2.jatjournal.org_article_10')

In [ ]:
with ThreadPoolExecutor(max_workers=5) as executor:
    tqdm(executor.map(download_pdf, pdf_urls), total=len(pdf_urls))

In [ ]:
files = [f.replace('.pdf', '') for f in os.listdir(f'pdfs/{pdfs_dir_name}/') if os.path.isfile(os.path.join(f'pdfs/{pdfs_dir_name}/', f))]
for e in pdf_urls:
    if e[1] not in files:
        print(e[1])

In [6]:
pdf_urls[0]

('https://www.e-revistes.uji.es/index.php/monti/article/download/1561/1295',
 'oai_IAO_article_1561')

In [14]:
download_pdf(pdf_urls[0])

Download failed. ('https://www.e-revistes.uji.es/index.php/monti/article/download/1561/1295', 'oai_IAO_article_1561')
